# Compact GAI 1D Residual + LightGBM Classifier

A standalone implementation that calculates Gut Age Index (GAI) residuals from scratch and uses them as a hybrid feature for CVD prediction.

## Workflow
1. **Setup & CLR**: Load CLR-transformed OTU data and metadata
2. **Age Predictor**: Train Ridge regression on healthy cohort
3. **Raw Residuals**: Calculate `raw_gai = predicted_age - true_age`
4. **Bias Correction**: Apply age-bin correction to normalize healthy cohort
5. **Hybrid Classification**: Stack corrected_gai with OTU features + LightGBM + 5-fold CV

In [ ]:
# ============================================================================
# SETUP: Imports and Data Loading Assumptions
# ============================================================================
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
import lightgbm as lgb

# ASSUMPTION: These DataFrames are already in memory:
# otu_clr      : DataFrame of shape (n_samples, n_otus) - CLR-transformed microbiome counts
# meta         : DataFrame with columns ['age', 'health', 'cardiovascular_disease']
#                where 'health' = 'y' for healthy, and cardiovascular_disease = 'I have this condition' for CVD
#
# If these don't exist in your notebook, run the previous cells to load and transform them.
# This cell assumes they are already available.

print("✓ Setup complete. Proceeding with GAI calculation and classification...")

In [ ]:
# ============================================================================
# COMPREHENSIVE: GAI 1D Residual + LightGBM Classification Pipeline
# ============================================================================
# This single cell implements the full workflow in 5 steps

print("=" * 80)
print("STEP 1: SETUP & DATA VALIDATION")
print("=" * 80)

# Assume otu_clr (n_samples × n_otus) and meta (n_samples × columns) are loaded
# Check data integrity
assert otu_clr.shape[0] == meta.shape[0], "Misaligned: otu_clr and meta have different sample counts"
n_samples, n_otus = otu_clr.shape
print(f"✓ Data loaded: {n_samples} samples × {n_otus} OTU features")
print(f"✓ Metadata columns: {meta.columns.tolist()}")

# Extract age and health status
age = meta['age'].values
is_healthy = meta['health'].values == 'y'
print(f"✓ Healthy samples: {is_healthy.sum()}, Non-healthy samples: {(~is_healthy).sum()}")

print("\n" + "=" * 80)
print("STEP 2: TRAIN AGE PREDICTOR ON HEALTHY COHORT ONLY")
print("=" * 80)

# Filter to only healthy individuals for training
X_healthy = otu_clr[is_healthy].values
y_healthy = age[is_healthy]

print(f"Training Ridge regressor on {len(X_healthy)} healthy samples...")
ridge = Ridge(alpha=1.0)  # Standard L2 regularization
ridge.fit(X_healthy, y_healthy)

# Calculate training MAE for reference
pred_healthy = ridge.predict(X_healthy)
mae_healthy = np.mean(np.abs(pred_healthy - y_healthy))
print(f"✓ Ridge regression trained. MAE on healthy cohort: {mae_healthy:.2f} years")

print("\n" + "=" * 80)
print("STEP 3: CALCULATE RAW RESIDUALS FOR ALL SAMPLES")
print("=" * 80)

# Predict "gut age" for ENTIRE dataset (both healthy and CVD)
predicted_age = ridge.predict(otu_clr.values)
raw_gai = predicted_age - age

print(f"Raw GAI statistics (all samples):")
print(f"  Mean: {raw_gai.mean():.4f}, Std: {raw_gai.std():.4f}")
print(f"  Min: {raw_gai.min():.2f}, Max: {raw_gai.max():.2f}")
print(f"✓ Raw GAI calculated for all {n_samples} samples")

print("\n" + "=" * 80)
print("STEP 4: AGE-BIN BIAS CORRECTION (Healthy Normalization)")
print("=" * 80)

# Define age bins exactly as specified
age_bins = [(18, 20), (20, 25), (25, 30), (30, 35), (35, 40), (40, 45), 
            (45, 50), (50, 55), (55, 60), (60, 65), (65, 70), (70, 75), (75, 100)]

# Calculate bin-specific means (using only healthy individuals)
bin_adjustments = {}
for start_age, end_age in age_bins:
    # Mask for healthy individuals in this age range
    mask_bin_healthy = is_healthy & (age >= start_age) & (age < end_age)
    
    if mask_bin_healthy.sum() > 0:
        bin_adjustments[(start_age, end_age)] = raw_gai[mask_bin_healthy].mean()
    else:
        bin_adjustments[(start_age, end_age)] = 0.0

print("Age-bin adjustment values (mean raw_gai per bin, healthy only):")
for (start_age, end_age), adj_val in sorted(bin_adjustments.items()):
    n_healthy_in_bin = (is_healthy & (age >= start_age) & (age < end_age)).sum()
    print(f"  [{start_age:2d}, {end_age:3d}) → adjustment: {adj_val:7.4f} (n={n_healthy_in_bin})")

# Apply bias correction to ALL samples (both healthy and CVD)
corrected_gai = raw_gai.copy()
for (start_age, end_age), adj_val in bin_adjustments.items():
    mask_bin = (age >= start_age) & (age < end_age)
    corrected_gai[mask_bin] = corrected_gai[mask_bin] - adj_val

print(f"\nCorrected GAI statistics (all samples):")
print(f"  Mean: {corrected_gai.mean():.4f}, Std: {corrected_gai.std():.4f}")
print(f"  Mean (healthy only): {corrected_gai[is_healthy].mean():.4f}  ← should be ~0")
print(f"✓ Bias correction applied. Healthy cohort now centered around 0.")

print("\n" + "=" * 80)
print("STEP 5: HYBRID STACKING & LIGHTGBM CLASSIFICATION")
print("=" * 80)

# Create hybrid feature matrix: CLR OTUs + corrected_gai
X_hybrid = otu_clr.copy()
X_hybrid['corrected_gai'] = corrected_gai

print(f"✓ Hybrid feature matrix created:")
print(f"  Shape: {X_hybrid.shape} ({n_otus} OTUs + 1 GAI feature)")

# Define target: CVD patients
# Expecting meta['cardiovascular_disease'] column with value 'I have this condition' for CVD
if 'cardiovascular_disease' in meta.columns:
    y = (meta['cardiovascular_disease'] == 'I have this condition').astype(int)
    print(f"  Target from 'cardiovascular_disease' column")
else:
    # Fallback: use 'health' column
    y = (meta['health'] != 'y').astype(int)
    print(f"  Target from 'health' column (health != 'y')")

print(f"✓ Target variable created:")
print(f"  No CVD (y=0): {(y == 0).sum()}")
print(f"  CVD (y=1): {(y == 1).sum()}")
print(f"  Class imbalance ratio: {(y == 0).sum() / (y == 1).sum():.1f}:1")

# 5-Fold Stratified Cross-Validation
print(f"\n5-Fold Stratified Cross-Validation with balanced LightGBM:")
print("─" * 80)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = []

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_hybrid, y), start=1):
    # Split data
    X_train, X_val = X_hybrid.iloc[train_idx], X_hybrid.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    # Initialize LightGBM with class_weight='balanced'
    clf = lgb.LGBMClassifier(
        class_weight='balanced',
        n_jobs=-1,
        num_leaves=31,
        learning_rate=0.1,
        random_state=42,
        verbose=-1
    )
    
    # Train and predict
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_val)
    y_pred_proba = clf.predict_proba(X_val)[:, 1]
    
    # Calculate metrics
    ba = balanced_accuracy_score(y_val, y_pred)
    auc = roc_auc_score(y_val, y_pred_proba)
    
    cv_results.append({'fold': fold_idx, 'balanced_accuracy': ba, 'auc_roc': auc})
    print(f"Fold {fold_idx}: Balanced Accuracy = {ba:.4f}, AUC-ROC = {auc:.4f}")

print("─" * 80)

# Aggregate results
cv_df = pd.DataFrame(cv_results)
mean_ba = cv_df['balanced_accuracy'].mean()
std_ba = cv_df['balanced_accuracy'].std()
mean_auc = cv_df['auc_roc'].mean()
std_auc = cv_df['auc_roc'].std()

print(f"\n✓ CROSS-VALIDATION RESULTS:")
print(f"  Balanced Accuracy: {mean_ba:.4f} ± {std_ba:.4f}")
print(f"  AUC-ROC:           {mean_auc:.4f} ± {std_auc:.4f}")

# Check target metrics
if mean_ba > 0.70 and mean_auc > 0.75:
    print(f"\n✓✓✓ TARGET METRICS MET! (BA > 70%, AUC > 75%)")
elif mean_ba > 0.70:
    print(f"\n✓ BA target met, but AUC target not reached")
elif mean_auc > 0.75:
    print(f"\n✓ AUC target met, but BA target not reached")
else:
    print(f"\n⚠ Target metrics not met. Consider hyperparameter tuning or feature engineering.")

print("\n" + "=" * 80)
print("✓ COMPLETE: GAI 1D Residual + LightGBM Pipeline")
print("=" * 80)

In [ ]:
# ============================================================================
# OPTIONAL: Inspect Corrected_GAI Feature
# ============================================================================
print("\nFINAL CORRECTED_GAI SUMMARY:")
print("=" * 80)

# Create output DataFrame with key features
gai_summary = pd.DataFrame({
    'sample_id': otu_clr.index,
    'age': age,
    'predicted_age': predicted_age,
    'raw_gai': raw_gai,
    'corrected_gai': corrected_gai,
    'is_healthy': is_healthy,
    'has_cvd': y.values
})

print(gai_summary.head(10))

print(f"\nStatistics by group:")
print(f"Healthy (y=0):")
print(f"  Corrected_GAI mean: {gai_summary[gai_summary['is_healthy']]['corrected_gai'].mean():.4f}")
print(f"  Corrected_GAI std:  {gai_summary[gai_summary['is_healthy']]['corrected_gai'].std():.4f}")
print(f"CVD (y=1):")
print(f"  Corrected_GAI mean: {gai_summary[~gai_summary['is_healthy']]['corrected_gai'].mean():.4f}")
print(f"  Corrected_GAI std:  {gai_summary[~gai_summary['is_healthy']]['corrected_gai'].std():.4f}")

print(f"\n✓ Corrected_GAI is now ready as a hybrid feature for downstream analysis!")

## Integration & Next Steps

### This Notebook vs. `hybrid_stacking_cvd.ipynb`

| Aspect | This Notebook | `hybrid_stacking_cvd.ipynb` |
|--------|---------------|---------------------------|
| **Scope** | Minimal, focused implementation | Comprehensive with SHAP interpretability |
| **Structure** | 3-4 self-contained cells | 21 cells with detailed sections |
| **Goal** | Quick prototyping / testing | Production-grade workflow |
| **Outputs** | CV metrics only | CV metrics + SHAP plots + visualizations |
| **Best For** | Experimentation, debugging | Full reproducible research pipeline |

### How to Use This Notebook

1. **In isolation**: Load your own `otu_clr` and `meta` dataframes. Run cells sequentially.
2. **With existing data**: Use the output of the CLR transformation cell from `hybrid_stacking_cvd.ipynb`, then run this notebook's cells.
3. **Custom integration**: Copy the main cell (Step 1-5) into your own workflow.

### Key Outputs

- `corrected_gai`: 1D continuous feature representing the "aging anomaly" bias-corrected
- `X_hybrid`: Feature matrix combining CLR-OTUs + corrected_gai
- **CV Metrics**: Balanced Accuracy and AUC-ROC for CVD prediction
- `gai_summary`: DataFrame with all intermediate calculations (raw_gai, predictions, etc.)

### Extensions

- Use `corrected_gai` alone (univariate) to compare against multivariate models
- Apply SHAP on the LightGBM model to rank importance of GAI vs. individual OTUs
- Export `corrected_gai` as a biomarker for clinical validation
- Stack corrected_gai predictions from other models (XGBoost, CatBoost, etc.)